# Notebook 03: EDA and Visualization

## Purpose

Create final-quality visualizations for the violation-level project. The focus is on how Boston building and housing violations vary by type, neighborhood, time, and location. We also include a compact RentSmart recent-case summary as external context.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

violations_path = PROCESSED_DIR / 'violations_clean.csv'
if not violations_path.exists():
    raise FileNotFoundError(f'Missing cleaned violations: {violations_path}. Run Notebook 02 first.')

df = pd.read_csv(violations_path, low_memory=False)
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
df['year'] = pd.to_numeric(df.get('year'), errors='coerce')
df['month'] = pd.to_numeric(df.get('month'), errors='coerce')
df['latitude'] = pd.to_numeric(df.get('latitude'), errors='coerce')
df['longitude'] = pd.to_numeric(df.get('longitude'), errors='coerce')
df['neighborhood'] = df['neighborhood'].fillna('Missing').astype(str)
df['description'] = df['description'].fillna('Missing').astype(str)

print(f'Loaded {len(df):,} cleaned violation records.')
print(f'Date range: {df["status_dttm"].min()} -> {df["status_dttm"].max()}')
df.head(3)

## 1. Top Violation Types

In [ ]:
top_types = df['description'].value_counts().head(15).reset_index()
top_types.columns = ['description', 'case_count']

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_types, x='case_count', y='description', color='#4C78A8', ax=ax)
ax.set_title('Top Building and Housing Violation Types')
ax.set_xlabel('Case count')
ax.set_ylabel('')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '01_top_violation_types.png', bbox_inches='tight')
plt.close(fig)
top_types

## 2. Violations by Neighborhood

In [ ]:
neighborhood_counts = (
    df[df['neighborhood'].ne('Missing')]['neighborhood']
    .value_counts()
    .head(20)
    .reset_index()
)
neighborhood_counts.columns = ['neighborhood', 'case_count']

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=neighborhood_counts, x='case_count', y='neighborhood', color='#59A14F', ax=ax)
ax.set_title('Building and Housing Violations by Neighborhood')
ax.set_xlabel('Case count')
ax.set_ylabel('')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_violations_by_neighborhood.png', bbox_inches='tight')
plt.close(fig)
neighborhood_counts

## 3. Violations Per Year

In [ ]:
year_counts = (
    df.dropna(subset=['year'])
    .assign(year=lambda x: x['year'].astype(int))
    .query('year >= 2010 and year <= 2026')
    .groupby('year')
    .size()
    .reset_index(name='case_count')
)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.lineplot(data=year_counts, x='year', y='case_count', marker='o', ax=ax)
ax.set_title('Building and Housing Violations per Year')
ax.set_xlabel('Year')
ax.set_ylabel('Case count')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_violations_per_year.png', bbox_inches='tight')
plt.close(fig)
year_counts.tail()

## 4. Monthly Seasonality

In [ ]:
month_counts = (
    df.dropna(subset=['month'])
    .assign(month=lambda x: x['month'].astype(int))
    .query('month >= 1 and month <= 12')
    .groupby('month')
    .size()
    .reset_index(name='case_count')
)

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.barplot(data=month_counts, x='month', y='case_count', color='#F28E2B', ax=ax)
ax.set_title('Monthly Seasonality of Violations')
ax.set_xlabel('Month')
ax.set_ylabel('Case count')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '04_monthly_seasonality.png', bbox_inches='tight')
plt.close(fig)
month_counts

## 5. Neighborhood-Year Heatmap

In [ ]:
heatmap_df = df[df['neighborhood'].ne('Missing')].dropna(subset=['year']).copy()
heatmap_df['year'] = heatmap_df['year'].astype(int)
top_neighborhoods = heatmap_df['neighborhood'].value_counts().head(12).index
heatmap_table = (
    heatmap_df[heatmap_df['neighborhood'].isin(top_neighborhoods)]
    .query('year >= 2018 and year <= 2025')
    .pivot_table(index='neighborhood', columns='year', values='case_no', aggfunc='count', fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_table, cmap='YlOrRd', linewidths=0.4, linecolor='white', ax=ax)
ax.set_title('Violation Counts by Neighborhood and Year')
ax.set_xlabel('Year')
ax.set_ylabel('Neighborhood')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_neighborhood_year_heatmap.png', bbox_inches='tight')
plt.close(fig)
heatmap_table

## 6. Spatial Distribution

In [ ]:
geo_df = df[
    df['latitude'].between(42.22, 42.40) & df['longitude'].between(-71.20, -70.98)
].dropna(subset=['latitude', 'longitude']).copy()

plot_df = geo_df.sample(n=min(6000, len(geo_df)), random_state=42)
fig, ax = plt.subplots(figsize=(7.5, 7))
ax.scatter(plot_df['longitude'], plot_df['latitude'], s=4, alpha=0.35, color='#4C78A8')
ax.set_title('Spatial Distribution of Building and Housing Violations')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '06_spatial_distribution.png', bbox_inches='tight')
plt.close(fig)
print(f'Plotted {len(plot_df):,} geocoded records.')

## 7. RentSmart Recent-Case Context

RentSmart is used as a supporting external context source. The raw current-case export is large, so the repository stores compact summaries by case type, neighborhood, and property type. This section visualizes the case-type distribution to check whether recent RentSmart patterns are consistent with the Building and Property Violations analysis.

In [ ]:
rentsmart_type_path = EXTERNAL_DIR / 'rentsmart_recent_cases_by_type.csv'
if rentsmart_type_path.exists():
    rentsmart_type = pd.read_csv(rentsmart_type_path)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    sns.barplot(data=rentsmart_type, x='case_count', y='violation_type', color='#4C78A8', ax=ax)
    ax.set_title('RentSmart Recent Cases by Type')
    ax.set_xlabel('Case count')
    ax.set_ylabel('')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '11_rentsmart_recent_cases_by_type.png', bbox_inches='tight')
    plt.close(fig)
    display(rentsmart_type)
else:
    print(f'Missing RentSmart summary: {rentsmart_type_path}')

## Summary

These visualizations show that violation records are concentrated by issue type, neighborhood, and time. The RentSmart current-case summary provides an external check that enforcement, housing, sanitation, and building-related cases are also important categories in a broader housing-quality data source.